# 🏢 Enterprise Lakehouse Catalog & Medallion Schema Setup
This notebook bootstraps the top-level **Unity Catalog** and provisions the core **Medallion Architecture schemas** (`bronze`, `silver`, `gold`) for the AtliQ FMCG Lakehouse.

### 📌 Step 1: Environment Bootstrap & Unity Catalog Initialization
* **Purpose:** Sets up the execution environment, initializes environment widgets, and creates/switches to the target Unity Catalog (`fmcg` in production, `fmcg_dev` in development).
* **Logic & Transformations:**
  1. Dynamically resolves repository root for modular imports (`src.compat`).
  2. Initializes Databricks compatibility context (`spark`, `dbutils`, `display`).
  3. Reads the `env` widget parameter (`prod` or `dev`).
  4. Executes `CREATE CATALOG IF NOT EXISTS` and `USE CATALOG` to establish the root catalog.
* **Inputs & Dependencies:** Interactive Databricks widget `env` (default: `prod`).
* **Outputs & Medallion State:** Active catalog switched to `fmcg` (or `fmcg_dev`).

In [1]:
# Initialize environment & Databricks compatibility (noop in Databricks)
import sys, os
try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(current_dir, "..")) if os.path.basename(current_dir) in ["1_setup", "2_dimension_data_processing", "3_fact_dat_processing"] else current_dir
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.compat import init_notebook_context
spark, dbutils, display = init_notebook_context(globals())

# Parameterize catalog via widget (supports fmcg or fmcg_dev)
dbutils.widgets.text("env", "prod", "Environment (dev/prod)")
env = dbutils.widgets.get("env").lower()
catalog_name = "fmcg_dev" if env == "dev" else "fmcg"
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"USE CATALOG {catalog_name}")


[Local Spark Emulation] Handled Unity Catalog command: CREATE CATALOG IF NOT EXISTS fmcg
[Local Spark Emulation] Handled Unity Catalog command: USE CATALOG fmcg


DataFrame[status: string]

### 📌 Step 2: Medallion Architecture Schema Provisioning
* **Purpose:** Creates the three primary Medallion layers (`bronze`, `silver`, `gold`) under the active catalog to isolate data by refinement and quality tiers.
* **Logic & Transformations:**
  * `bronze`: Stores raw, append-only landing data preserving AWS S3 metadata and Change Data Feed.
  * `silver`: Stores cleansed, standardized, deduplicated, and conformed master entities.
  * `gold`: Stores high-performance dimensional star schema models (facts & dimensions) optimized for analytics and BI.
* **Inputs & Dependencies:** Target `catalog_name` established in Step 1.
* **Outputs & Medallion State:** Unity Catalog schemas `{catalog_name}.bronze`, `{catalog_name}.silver`, and `{catalog_name}.gold` provisioned.

In [2]:
# Create medallion schemas under the target catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.bronze")


[Local Spark Emulation] Multi-part namespace adapted: CREATE SCHEMA IF NOT EXISTS fmcg.gold -> CREATE SCHEMA IF NOT EXISTS gold
[Local Spark Emulation] Multi-part namespace adapted: CREATE SCHEMA IF NOT EXISTS fmcg.silver -> CREATE SCHEMA IF NOT EXISTS silver
[Local Spark Emulation] Multi-part namespace adapted: CREATE SCHEMA IF NOT EXISTS fmcg.bronze -> CREATE SCHEMA IF NOT EXISTS bronze


DataFrame[]

### 📌 Step 3: Catalog Health & Metastore Smoke Test
* **Purpose:** Verifies that metastore connectivity is established and checks for existing dimensional tables (`dim_customers`) to confirm Lakehouse readiness.
* **Logic & Transformations:** Checks `spark.catalog.tableExists()` for `gold.dim_customers`. If already populated, queries current record count; otherwise, logs a readiness confirmation for subsequent ingestion pipelines.
* **Inputs & Dependencies:** Provisioned schemas from Step 2.
* **Outputs & Medallion State:** Status log message or customer record count display.

In [3]:
# Smoke test verification (safe check if table exists)
if spark.catalog.tableExists(f"{catalog_name}.gold.dim_customers"):
    display(spark.sql(f"SELECT COUNT(*) as customer_count FROM {catalog_name}.gold.dim_customers"))
else:
    print(f"Catalog {catalog_name} and schemas initialized. dim_customers table will be created during dimensional ingestion.")


Catalog fmcg and schemas initialized. dim_customers table will be created during dimensional ingestion.
